In [ ]:
import pandas as pd
import os

def scan_and_process_sales_employees():
    path = '/content/'
    files = [f for f in os.listdir(path) if f.endswith('.csv')]

    # Schema mục tiêu cho SALES_EMPLOYEE
    target_cols = ['sales_employee_id', 'employee_name', 'marital_status', 'education_level', 'years_experience']

    # Mapping dự đoán các tên cột tương đồng
    potential_mappings = {
        'sales_employee_id': ['sales_employee_id', 'employee_id', 'staff_id', 'emp_id'],
        'employee_name': ['employee_name','sales_employee_name', 'name', 'full_name', 'sales_rep_name'],
        'marital_status': ['marital_status', 'marital', 'marriage'],
        'education_level': ['education_level', 'education', 'degree_level'],
        'years_experience': ['years_experience', 'experience', 'experience_years', 'tenure']
    }

    source_report = {}
    collected_dfs = []

    print("--- Bắt đầu quét các file cho bảng SALES_EMPLOYEE ---")
    for file in files:
        if file == 'sales_employee_new.csv': continue
        try:
            file_path = os.path.join(path, file)
            # Đọc header với utf-8-sig
            temp_df = pd.read_csv(file_path, nrows=0, encoding='utf-8-sig')
            found_cols = {}

            for target, aliases in potential_mappings.items():
                for alias in aliases:
                    if alias in temp_df.columns:
                        found_cols[alias] = target
                        break

            # Cần ít nhất ID hoặc Tên để nhận diện bảng nhân viên
            if 'sales_employee_id' in found_cols.values() or 'employee_name' in found_cols.values():
                print(f"Tìm thấy dữ liệu SALES_EMPLOYEE trong: {file} ({list(found_cols.keys())})")
                full_df = pd.read_csv(file_path, encoding='utf-8-sig')
                mapped_df = full_df[list(found_cols.keys())].rename(columns=found_cols)
                collected_dfs.append(mapped_df)

                for alias, target in found_cols.items():
                    if target not in source_report:
                        source_report[target] = []
                    source_report[target].append(f"{file} (gốc: {alias})")
        except Exception:
            continue

    if not collected_dfs:
        print("Không tìm thấy file nào chứa thông tin nhân viên bán hàng.")
        return

    # Gộp và xóa trùng lặp dựa trên Primary Key (sales_employee_id)
    final_df = pd.concat(collected_dfs, ignore_index=True)
    initial_len = len(final_df)
    if 'sales_employee_id' in final_df.columns:
        final_df = final_df.drop_duplicates(subset=['sales_employee_id'], keep='first')
    else:
        final_df = final_df.drop_duplicates()
        
    print(f"\nĐã xử lý: Xóa {initial_len - len(final_df)} dòng trùng lặp.")

    # Đảm bảo schema và điền mặc định
    for col in target_cols:
        if col not in final_df.columns:
            final_df[col] = None

    defaults = {
        'sales_employee_id': 'UNKNOWN_EMP',
        'employee_name': 'Unknown',
        'marital_status': 'Unknown',
        'education_level': 'Unknown',
        'years_experience': 0
    }
    for col, val in defaults.items():
        final_df[col] = final_df[col].fillna(val)

    # Lưu kết quả
    output_file = '/content/sales_employee_new.csv'
    final_df[target_cols].to_csv(output_file, index=False, encoding='utf-8-sig')

    print(f"\n--- HOÀN THÀNH ---")
    print(f"File lưu tại: {output_file}")

    print("\n--- BÁO CÁO NGUỒN DỮ LIỆU (SOURCE REPORT) ---")
    for col in target_cols:
        sources = ", ".join(source_report.get(col, ["Không tìm thấy - Sử dụng mặc định"]))
        print(f"Cột '{col}': {sources}")

scan_and_process_sales_employees()